# Code mẫu — MLP và backpropagation

MLP tạo ra các đặc trưng trung gian ở lớp ẩn, rồi kết hợp chúng ở lớp đầu ra. Ta bắt đầu từ bài XOR với trọng số đặt bằng tay. Sau đó kiểm backprop bằng ba cách tính khác nhau, và cuối cùng cho một mạng thật tự học bốn mẫu.

## Ví dụ 1: Tạo đặc trưng ẩn để giải XOR

Hai đầu vào là trạng thái của hai công tắc, mỗi cái 0 hoặc 1. Nhãn bằng 1 khi có đúng một công tắc bật. Ta đặt hai đặc trưng ẩn $h_1=\mathrm{ReLU}(x_1+x_2)$ và $h_2=\mathrm{ReLU}(x_1+x_2-1)$, rồi lấy đầu ra là $h_1-2h_2$.

In [ ]:
import numpy as np

X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
target = np.array([0., 1., 1., 0.])
W1 = np.ones((2, 2))  # Mỗi cột tạo ra một đặc trưng ẩn.
b1 = np.array([0., -1.])
Z = X @ W1 + b1
hidden = np.maximum(Z, 0.)
prediction = hidden @ np.array([1., -2.])
print("hidden:", hidden.tolist())
print("XOR:", prediction.tolist())
print("bỏ ReLU:", (Z @ np.array([1., -2.])).tolist())
assert np.array_equal(prediction, target)

hidden: [[0.0, 0.0], [1.0, 0.0], [1.0, 0.0], [2.0, 1.0]]
XOR: [0.0, 1.0, 1.0, 0.0]
bỏ ReLU: [2.0, 1.0, 1.0, 0.0]


Tổng hai đầu vào chỉ có thể là 0, 1 hoặc 2. Ứng với ba trường hợp đó, lớp ẩn cho `(0, 0)`, `(1, 0)`, `(2, 1)`, và đầu ra là 0, 1, 0. Đúng bảng XOR.

Chỗ quyết định là ReLU. Nó cắt $h_2$ về 0 khi chỉ có một công tắc bật, nên $h_2$ chỉ “lên tiếng” khi cả hai cùng bật. Bỏ ReLU đi, cả mạng chỉ còn là một phép tuyến tính, và không khớp được bốn nhãn. Lưu ý là các trọng số ở đây do ta thiết kế, mạng chưa học gì cả.

## Ví dụ 2: Tính gradient bằng tay rồi đối chiếu bằng hai cách khác

Hai mẫu, mỗi mẫu có hai đặc trưng, đi qua mạng `2 → 2 → 1` để dự đoán một số thực. Loss là MSE, lấy trung bình trên hai mẫu. Dự đoán là 1.75 và 0.25, target là 1 và 0, nên loss bằng $(0.75^2+0.25^2)/2=0.3125$. Ta tính gradient của cả bốn nhóm tham số theo đúng các công thức backprop.

In [ ]:
import numpy as np
import torch

torch.set_num_threads(1)
X = np.array([[1., 0.], [0., 2.]])
y = np.array([[1.], [0.]])
params = [np.array([[1., -1.], [.5, 1.]]), np.array([.5, -.5]),
          np.array([[1.], [-1.]]), np.array([.25])]

def forward(ps):
    W1, b1, W2, b2 = ps
    z = X @ W1 + b1
    h = np.maximum(z, 0.)
    return h @ W2 + b2, z, h

def objective(ps):
    p, _, _ = forward(ps)
    assert p.shape == y.shape
    return ((p - y) ** 2).mean()

prediction, z, h = forward(params)
g_output = 2 * (prediction - y) / len(X)
# Dùng W2 của chính lần forward này. Chưa tham số nào được cập nhật.
g_hidden = (g_output @ params[2].T) * (z > 0)
manual = [X.T @ g_hidden, g_hidden.sum(axis=0),
          h.T @ g_output, g_output.sum(axis=0)]
print("z:", z.tolist())
print("prediction, MSE:", prediction.ravel().tolist(), objective(params))
for name, gradient in zip(["W1", "b1", "W2", "b2"], manual):
    print("gradient", name, np.round(gradient, 4).tolist())

z: [[1.5, -1.5], [1.5, 1.5]]
prediction, MSE: [1.75, 0.25] 0.3125
gradient W1 [[0.75, 0.0], [0.5, -0.5]]
gradient b1 [1.0, -0.25]
gradient W2 [[1.5], [0.375]]
gradient b2 [1.0]


Ở mẫu đầu, neuron ẩn thứ hai có $z=-1.5$. ReLU trả về 0 ở đó, nên gradient không đi qua được nhánh này. Vì vậy ô `W1[0][1]` có gradient bằng 0. Hàng đầu của gradient `W2` là $1.5\cdot0.75+1.5\cdot0.25=1.5$. Hai mẫu cùng dùng một tham số, nên phần đóng góp của chúng được cộng lại.

Bây giờ kiểm lại bằng hai cách độc lập. Cách thứ nhất là sai phân trung tâm: nhích từng tham số lên một chút, nhích xuống một chút, rồi xem loss đổi bao nhiêu. Cách thứ hai là để autograd của PyTorch tự tính. Mọi giá trị `z` ở đây đều cách xa 0, nên ta không vô tình đo đạo hàm ngay tại chỗ gãy của ReLU.

In [ ]:
numeric = []
eps = 1e-5
for parameter in params:
    gradient = np.zeros_like(parameter)
    for index in np.ndindex(parameter.shape):
        old = parameter[index]
        parameter[index] = old + eps
        plus = objective(params)
        parameter[index] = old - eps
        minus = objective(params)
        parameter[index] = old
        gradient[index] = (plus - minus) / (2 * eps)
    numeric.append(gradient)

tp = [torch.tensor(p, dtype=torch.float64, requires_grad=True) for p in params]
tx, ty = torch.tensor(X), torch.tensor(y)
pred = torch.relu(tx @ tp[0] + tp[1]) @ tp[2] + tp[3]
(pred - ty).square().mean().backward()
numeric_ok = all(np.allclose(a, b, atol=1e-8) for a, b in zip(manual, numeric))
autograd_ok = all(np.allclose(g, p.grad.numpy(), atol=1e-10)
                  for g, p in zip(manual, tp))
updated = [p - .05 * g for p, g in zip(params, manual)]
print("sai phân khớp:", numeric_ok, "autograd khớp:", autograd_ok)
print("loss trước / sau một bước:", round(objective(params), 6),
      round(objective(updated), 6))
assert numeric_ok and autograd_ok and objective(updated) < objective(params)

sai phân khớp: True autograd khớp: True
loss trước / sau một bước: 0.3125 0.132988


Ba cách tính đi theo ba con đường khác nhau mà cho cùng một gradient. Đây là cách đáng tin nhất để kiểm một bản backprop tự viết. Cập nhật mọi tham số một bước, sau khi đã tính xong toàn bộ gradient, thì loss giảm từ 0.3125 xuống còn khoảng 0.133. Phép kiểm này chỉ xác nhận cơ chế tính đúng tại một điểm, chưa nói gì về việc model dự đoán tốt trên dữ liệu mới.

## Ví dụ 3: Một MLP tự học bốn mẫu từ khởi tạo ngẫu nhiên

Vẫn là bài XOR, nhưng lần này không đặt trọng số bằng tay. Ta dùng mạng `2 → 8 → 1` với hàm kích hoạt Tanh, khởi tạo ngẫu nhiên, và để nó tự học. Đầu ra là một logit cho bài nhị phân. `BCEWithLogitsLoss` nhận thẳng logit và nhãn kiểu số thực. Cả bốn mẫu nằm trong một batch. Cho model học thuộc một batch nhỏ như vậy là cách kiểm nhanh xem model và vòng lặp train có hoạt động không.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(31)
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([0., 1., 1., 0.])
model = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=.05)
criterion = nn.BCEWithLogitsLoss()
print("tham số:", sum(p.numel() for p in model.parameters()))
for step in range(301):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(X).squeeze(-1)  # Chỉ bỏ trục đầu ra, giữ nguyên trục batch.
    assert logits.shape == y.shape
    loss = criterion(logits, y)
    if step in [0, 50, 150, 300]:
        print("step", step, "loss", round(loss.item(), 5))
    if step < 300:
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    probabilities = model(X).squeeze(-1).sigmoid()
    labels = (probabilities >= .5).long()
print("xác suất:", [round(p, 4) for p in probabilities.tolist()])
print("nhãn:", labels.tolist())
assert torch.equal(labels, y.long())

tham số: 33
step 0 loss 0.69369
step 50 loss 0.00815
step 150 loss 0.00119
step 300 loss 0.00047
xác suất: [0.0004, 0.9998, 0.9994, 0.0007]
nhãn: [0, 1, 1, 0]


Mạng có 33 tham số: lớp đầu có $2\cdot8+8=24$, lớp sau có $8+1=9$. Loss ban đầu là 0.6937, gần bằng $\ln 2$, đúng mức của một model chưa biết gì và đoán 50–50. Sau 300 bước, loss gần như bằng 0, và nhãn đoán ra khớp `[0, 1, 1, 0]`.

Sigmoid chỉ được dùng lúc đọc xác suất, không dùng trong loss, vì `BCEWithLogitsLoss` đã tự lo phần đó. Kết quả này chỉ cho thấy mạng học được đúng bốn mẫu nó đã thấy. Muốn biết mạng làm tốt tới đâu trên dữ liệu mới, phải có một tập riêng. Đừng gọi accuracy trên chính batch train là validation.